# Clearvale × Kim et al. (2020): the method, in figures (sub-090)

Recomputes the feature-selection GLMs of `config-cv-kim2020-denoised.json` for one subject with the pipeline's own functions (`workflows/template_rsa.py`, `betaestimation`), and draws:

1. **The method, step by step** -- and where it switches from GLM t-maps to normalized, lag-shifted BOLD.
2. **The two GLMs** -- design matrices, runs stacked into one model.
3. **Event timing** -- which volumes become a template and which a target timecourse.
4. **Contrast maps** on the subject's inflated cortex (category vs. others, whole brain).
5. **Two items' representations** -- weights, template, and their WM timecourses.

Run from top to bottom (about 3-5 minutes); figures are also saved to `figures/`.

In [ ]:
import os, sys, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import nibabel as nib
from nilearn import image, plotting, surface
from nilearn.maskers import NiftiMasker

REPO = Path.cwd().resolve().parents[2]  # run from examples/clearvale/notebooks
HERE = Path(os.environ.get("CV_BIDS", REPO / "data" / "cv-bids"))  # the Clearvale BIDS folder
os.chdir(REPO)                      # config paths are relative to the repo root
sys.path.insert(0, str(REPO))
from betaestimation.glm import concatenated_design, fit_concatenated, contrast
from betaestimation.timeseries import window_volumes
from utils.rsa_common import load_config, read_master_spreadsheet, resolve_roi_mask, evaluate_query_node
from workflows import template_rsa as T

CONFIG = REPO / "examples" / "clearvale" / "configs" / "config-cv-kim2020-denoised.json"
SUBJECT = "090"
FIG = HERE / "figures"; FIG.mkdir(exist_ok=True)
SURF = HERE / "derivatives/bids-hcp/sub-090/ses-A1/MNINonLinear/fsaverage_LR32k"

cfg = load_config(str(CONFIG))
df = read_master_spreadsheet(cfg["event_extraction"]["output_file"])
sub_df = df[df["subject"] == SUBJECT].copy()
tr_cfg = cfg["template_rsa"]
tmpl_events = T.select(sub_df, tr_cfg["templates"]["events"])
ex_col = T.item_column(cfg); cat_col, sub_col = T.template_levels(cfg)
items = (pd.DataFrame({"item": tmpl_events[ex_col].astype(str), "category": tmpl_events[cat_col].astype(str),
                       "subcategory": tmpl_events[sub_col].astype(str)}).drop_duplicates("item").reset_index(drop=True))
loc_bolds = sorted(tmpl_events["boldfile"].unique())
wm_bolds = sorted(sub_df.loc[sub_df["task"].str.startswith("WM"), "boldfile"].unique())
print(f"{len(items)} items ({items['category'].value_counts().to_dict()}), {len(loc_bolds)} localizer runs, {len(wm_bolds)} WM runs")

## 1. The method, and where GLM t-maps end and BOLD begins

**The GLMs are only used to choose and weight voxels.** Every pattern that is ever correlated -- each template and each target -- is the *z-scored, lag-shifted BOLD signal*, never a beta or t-map. The t-maps enter twice, and only as voxel selections / multipliers:

| step | input | what comes out | GLM or BOLD? |
|---|---|---|---|
| A. category mask | localizer runs, **one GLM over all 3 runs stacked**, one regressor per category (6-image mini-blocks merged) | category-vs-others **z map**, thresholded p < .05 (one-sided), clusters ≥ 10 voxels → a binary mask per category | **GLM** |
| B. item weights | localizer runs, **one LSA GLM over all 3 runs stacked**, one regressor per image (90) | each image's item-vs-others **t map** → voxel weights | **GLM** |
| C. template | the image's 3 localizer item instances: **z-scored BOLD volumes** from onset+4.6 s to offset+4.6 s, averaged over volumes and item instances | a pattern, restricted to the category mask (A) and multiplied by the weights (B) | **BOLD** (shaped by A, B) |
| D. target | each WM view event: **every z-scored BOLD volume** from view onset to +13.8 s (30 TRs, unshifted) | each volume × the same image's weights (B), within its mask (A) | **BOLD** (shaped by A, B) |
| E. score | template (C) vs. target volume (D) | Pearson r → Fisher z, per TR, for target / related / others / switch sets | -- |

So the switch happens between B and C: the t-maps are computed once, from the localizer, and from then on the analysis only works with normalized time series (detrended, 128 s high-pass, z-scored per run, FD > 0.9 mm volumes and the first 10 volumes set to NaN).

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.4)); ax.axis("off"); ax.set_xlim(0, 15); ax.set_ylim(0, 6.4)
def box(x, y, w, h, text, color):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", fc=color, ec="k", lw=1.2, zorder=3))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=9.5, zorder=4)
def arrow(p, q, color="k", rad=0.0):
    ax.annotate("", q, p, arrowprops=dict(arrowstyle="->", lw=1.4, color=color, connectionstyle=f"arc3,rad={rad}"), zorder=2)
GLM, BOLD, OUT = "#f6d7b0", "#bfe0f5", "#d5f0c8"
ax.add_patch(mpatches.Rectangle((0.1, 0.1), 7.0, 6.2, fc="#fff7ec", ec="#e0a050", ls="--", zorder=0))
ax.text(0.25, 6.0, "GLM t-maps: feature selection only", color="#b06000", fontsize=10.5, weight="bold")
ax.add_patch(mpatches.Rectangle((7.3, 0.1), 7.6, 6.2, fc="#f0f8ff", ec="#5090c0", ls="--", zorder=0))
ax.text(7.45, 6.0, "z-scored, lag-shifted BOLD: every pattern that is correlated", color="#205080", fontsize=10.5, weight="bold")
box(0.3, 4.55, 2.5, 1.05, "Localizer: 3 runs\n90 images x 3\n(1.85 s each)", "white")
box(3.4, 3.35, 3.5, 1.0, "A. Category GLM\n(3 runs stacked)\nz > 1.645, k >= 10  ->  masks", GLM)
box(3.4, 1.95, 3.5, 1.0, "B. Item LSA GLM\n(3 runs stacked, 90 regressors)\nitem vs. others t  ->  weights", GLM)
box(0.3, 0.35, 2.5, 1.05, "WM: 8 runs (WMpos/neg)\n36 trials: view ->\noperation -> fixation", "white")
box(7.6, 4.55, 3.1, 1.05, "Localizer BOLD, z-scored\nonset+4.6 .. offset+4.6 s\nmean of volumes & 3 reps", BOLD)
box(7.6, 0.35, 3.1, 1.05, "WM BOLD, z-scored\nevery TR from view onset\nto +13.8 s (30 TRs)", BOLD)
box(11.4, 4.55, 3.3, 1.05, "C. Template (image i)\n= pattern x weights_i\nwithin mask_category(i)", OUT)
box(11.4, 0.35, 3.3, 1.05, "D. Target (each volume)\n= pattern x weights_i\nwithin mask_category(i)", OUT)
box(11.4, 2.4, 3.3, 1.15, "E. Pearson r -> Fisher z\ntarget / related / others\nswitch_item / switch_category", "#eeeeee")
arrow((2.8, 5.08), (7.6, 5.08)); arrow((2.8, 0.88), (7.6, 0.88))
arrow((1.55, 4.55), (3.4, 3.85), rad=0.15); arrow((1.55, 4.55), (3.4, 2.45), rad=0.25)
arrow((10.7, 5.08), (11.4, 5.08)); arrow((10.7, 0.88), (11.4, 0.88))
arrow((13.05, 4.55), (13.05, 3.55)); arrow((13.05, 1.4), (13.05, 2.4))
# masks and weights leave the GLM panel and shape both the templates and the targets
for y0 in (3.85, 2.45):
    arrow((6.9, y0), (11.4, 4.75), color="#b06000", rad=0.12)
    arrow((6.9, y0), (11.4, 1.2), color="#b06000", rad=-0.12)
ax.text(8.0, 3.05, "masks + weights\n(computed once, from the localizer)", color="#b06000", fontsize=9, ha="left")
ax.set_title("Kim et al. (2020) template RSA as configured for Clearvale", fontsize=13, weight="bold")
fig.savefig(FIG / "1_method_flow.png", dpi=150, bbox_inches="tight"); plt.show()

## 2. The two feature-selection GLMs: runs stacked, not averaged

Both GLMs are **one model over the three localizer runs stacked in time** (SPM's multi-session model): the regressors of interest (`cat_*` / `exm_*`) are *shared* across runs, so each image's three item instances estimate one beta; everything else -- the run's own intercept, cosine drifts, motion and the other events (rest, unselected trials) -- is *run-specific* (block-diagonal), so runs keep their own baselines. Censored volumes (first 10, FD > 0.9 mm) are dropped as rows. Nothing is averaged across runs before the fit.

In [ ]:
ROI = tr_cfg["rois"][0]
roi_img = resolve_roi_mask(cfg["rois"][ROI], SUBJECT, loc_bolds[0])
roi_masker = NiftiMasker(mask_img=roi_img).fit()
runs = {b: T.load_run(cfg, sub_df[sub_df["boldfile"] == b], roi_masker) for b in loc_bolds + wm_bolds}
print(f"ROI {ROI}: {int(np.asarray(roi_img.dataobj).sum())} voxels; loaded {len(runs)} runs")

fw = tr_cfg["feature_weighting"]; cm = fw["category_mask"]
def specs(events_by_run):
    return [{"keep": runs[b].keep, "n_frames": runs[b].n_frames, "tr": runs[b].tr, "events": ev,
             "confounds": runs[b].confounds} for b, ev in events_by_run.items()]
cat_events = T.select(sub_df, cm.get("events") or tr_cfg["templates"]["events"])
cat_ev = {b: T.glm_events(r, r.index.isin(cat_events.index), "cat_" + r[cat_col].astype(str),
                          cm.get("block_merge_gap_seconds")) for b, r in sub_df[sub_df["boldfile"].isin(loc_bolds)].groupby("boldfile")}
exm_ev = {b: T.glm_events(r, r.index.isin(tmpl_events.index), "exm_" + r[ex_col].astype(str).map(T.safe_name))
          for b, r in sub_df[sub_df["boldfile"].isin(loc_bolds)].groupby("boldfile")}
X_cat = concatenated_design(specs(cat_ev), "cat_", cfg["pattern_estimation"]["glm"])
X_exm = concatenated_design(specs(exm_ev), "exm_", cfg["pattern_estimation"]["glm"])
run_rows = np.cumsum([len(runs[b].keep) for b in loc_bolds])
print("category GLM:", X_cat.shape, "| item GLM:", X_exm.shape, "| rows per run:", np.diff(np.r_[0, run_rows]))

In [ ]:
def group(col):
    c = str(col)
    if c.startswith(("cat_", "exm_")): return "interest (shared)"
    c = re.sub(r"^run\d+_", "", c)
    if c.startswith("drift") or c == "constant": return "drift + intercept (per run)"
    if c in ("trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"): return "motion (per run)"
    return "other events (per run)"
order = ["interest (shared)", "other events (per run)", "motion (per run)", "drift + intercept (per run)"]
def plot_design(X, ax, title):
    cols = sorted(X.columns, key=lambda c: (order.index(group(c)), str(c)))
    M = X[cols].to_numpy(dtype=float)
    M = M / np.maximum(np.abs(M).max(axis=0), 1e-12)
    ax.imshow(M, aspect="auto", cmap="gray_r", interpolation="nearest", vmin=-1, vmax=1)
    for r in run_rows[:-1]: ax.axhline(r - 0.5, color="red", lw=1)
    start, k = 0, 0
    for g in order:
        n = sum(group(c) == g for c in cols)
        if n:
            ax.axvline(start - 0.5, color="tab:blue", lw=0.8)
            ax.text(start + n / 2, 1.005 + 0.055 * (k % 2), f"{g} ({n})", ha="center", va="bottom", fontsize=8,
                    transform=ax.get_xaxis_transform())
            k += 1
        start += n
    ax.set_title(title, pad=44); ax.set_xlabel("regressors"); ax.set_ylabel("kept volumes, runs stacked (red = run boundary)")
    ax.set_xticks([])
fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
plot_design(X_cat, axes[0], "A. Category-mask GLM (6-image mini-blocks merged)")
plot_design(X_exm, axes[1], "B. Item LSA GLM (90 image regressors)")
fig.tight_layout(); fig.savefig(FIG / "2_design_matrices.png", dpi=150, bbox_inches="tight"); plt.show()

# zoom: the regressors of interest over the first 90 s of run 1
t = np.arange(len(runs[loc_bolds[0]].keep))[:200] * runs[loc_bolds[0]].tr
fig, axes = plt.subplots(2, 1, figsize=(15, 5), sharex=True)
for c in [c for c in X_cat.columns if str(c).startswith("cat_")]:
    axes[0].plot(t, X_cat[c].to_numpy()[:200], label=c)
axes[0].legend(ncol=3, fontsize=8); axes[0].set_title("A. category regressors, run 1 (one boxcar per mini-block)")
first = [c for c in X_exm.columns if str(c).startswith("exm_") and X_exm[c].to_numpy()[:200].any()]
for c in first: axes[1].plot(t, X_exm[c].to_numpy()[:200], lw=1)
axes[1].set_title(f"B. item regressors, run 1: {len(first)} images in these 90 s, one regressor each"); axes[1].set_xlabel("time from first kept volume (s)")
fig.tight_layout(); fig.savefig(FIG / "2b_regressors_zoom.png", dpi=150, bbox_inches="tight"); plt.show()

## 3. Which volumes become a template, and which a target

Top: the first 90 s of localizer run 1. Each image (1.85 s) contributes the volumes from onset + 4.6 s to offset + 4.6 s (blue) to its template; three such windows (one per run) are averaged. Bottom: one WM trial; the target timecourse is every TR from view onset to +13.8 s (30 TRs), *not* shifted -- the hemodynamic delay shows up as the peak around 5-8 s.

In [ ]:
def timeline(ax, ev, t_max, shade=None, title=""):
    colors = {"view": "tab:orange", "face": "tab:red", "place": "tab:green", "item": "tab:purple", "rest": "lightgray"}
    for _, e in ev[ev["onset"] < t_max].iterrows():
        tt = str(e["trial_type"]); c = next((v for k, v in colors.items() if k in tt), "tab:gray")
        if tt.startswith(("maintain", "suppress", "clear", "switch")): c = "tab:blue"
        ax.add_patch(mpatches.Rectangle((e["onset"], 0.55), e["duration"], 0.4, fc=c, ec="k", lw=0.4))
        if e["duration"] > 1.5: ax.text(e["onset"] + e["duration"] / 2, 1.0, tt.replace("_", "\n"), ha="center", va="bottom", fontsize=6.5)
    if shade is not None:
        for a, b in shade: ax.add_patch(mpatches.Rectangle((a, 0.05), b - a, 0.4, fc="tab:blue", alpha=0.35))
    tr = 0.46
    ax.vlines(np.arange(0, t_max, tr), 0.0, 0.05, color="k", lw=0.3)
    ax.set_xlim(0, t_max); ax.set_ylim(0, 1.35); ax.set_yticks([]); ax.set_title(title); ax.set_xlabel("time in run (s)")
loc_ev = pd.read_csv(HERE / "sub-090/ses-A1/func/sub-090_ses-A1_task-loc_dir-pa_run-01_events.tsv", sep="\t", na_values="n/a")
wm_ev = pd.read_csv(HERE / "sub-090/ses-A1/func/sub-090_ses-A1_task-WMpos_dir-pa_run-01_events.tsv", sep="\t", na_values="n/a")
imgs = loc_ev[loc_ev["stimulus"].notna()]
shade = [(o + 4.6, o + d + 4.6) for o, d in zip(imgs["onset"], imgs["duration"])]
v = wm_ev[wm_ev["trial_type"].str.startswith("view")].iloc[0]
fig, axes = plt.subplots(2, 1, figsize=(15, 5))
timeline(axes[0], loc_ev, 90, shade, "Localizer run 1: images (top) and their template windows, onset+4.6 .. offset+4.6 s (blue); ticks = TRs")
timeline(axes[1], wm_ev[(wm_ev["onset"] >= v["onset"] - 3) & (wm_ev["onset"] < v["onset"] + 16)].assign(onset=lambda d: d["onset"]), v["onset"] + 16,
         [(v["onset"], v["onset"] + 13.8)], f"WM run 1, first trial ({v['stimulus']}, {v['operation']}): target timecourse = 30 TRs from view onset (blue)")
axes[1].set_xlim(v["onset"] - 3, v["onset"] + 16)
fig.tight_layout(); fig.savefig(FIG / "3_event_timing.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Category contrast maps on the inflated cortex

For display only, the category GLM (A) is refit **across the whole brain** (the analysis itself fits it inside the VVS ROI, which gives the same values there). Shown: category-vs-others t maps, thresholded at t > 3.1, projected from the MNI volume onto the subject's own HCP midthickness surface and drawn on its inflated surface (ventral and lateral views). The black outline is the VVS ROI.

In [ ]:
brain = image.resample_to_img(str(HERE / "derivatives/bids-hcp/sub-090/ses-A1/MNINonLinear/brainmask_fs.2.nii.gz"),
                              loc_bolds[0], interpolation="nearest", force_resample=True, copy_header=True)
brain_masker = NiftiMasker(mask_img=image.math_img("img > 0", img=brain)).fit()
wb_runs = {b: T.load_run(cfg, sub_df[sub_df["boldfile"] == b], brain_masker) for b in loc_bolds}
wb_specs = [{"data": wb_runs[b].glm, "keep": wb_runs[b].keep, "n_frames": wb_runs[b].n_frames, "tr": wb_runs[b].tr,
             "events": cat_ev[b], "confounds": wb_runs[b].confounds} for b in loc_bolds]
labels, results, cols = fit_concatenated(wb_specs, "cat_", cfg["pattern_estimation"]["glm"])
cats = ["face", "place", "item"]
tmaps = {}
for c in cats:
    others = [o for o in cats if o != c]
    w = {f"cat_{c}": 1.0, **{f"cat_{o}": -1.0 / len(others) for o in others}}
    tmaps[c] = brain_masker.inverse_transform(contrast(labels, results, cols, w, "t"))
    tmaps[c].to_filename(FIG / f"sub-090_category-{c}_vs_others_t.nii.gz")
print("whole-brain voxels:", wb_runs[loc_bolds[0]].clean.shape[1])

In [ ]:
mesh = {h: str(SURF / f"090.{h}.midthickness.32k_fs_LR.surf.gii") for h in "LR"}
infl = {h: str(SURF / f"090.{h}.inflated.32k_fs_LR.surf.gii") for h in "LR"}
sulc = {h: str(SURF / f"090.{h}.sulc.32k_fs_LR.shape.gii") for h in "LR"}
def to_surf(img, h, interp="linear"):
    return surface.vol_to_surf(img, mesh[h], interpolation=interp, radius=3.0)
# VVS outline from the same Desikan-Killiany regions as the ROI, on the subject's surface parcellation
VVS_NAMES = ("fusiform", "inferiortemporal", "lateraloccipital", "lingual", "parahippocampal")
def surf_roi(h):
    g = nib.load(str(SURF / f"090.{h}.aparc.32k_fs_LR.label.gii"))
    keys = [k for k, v in g.labeltable.get_labels_as_dict().items() if v.split("_", 1)[-1] in VVS_NAMES]
    return np.isin(g.darrays[0].data, keys).astype(int)
roi_surf = {h: surf_roi(h) for h in "LR"}
hemi = {"L": "left", "R": "right"}
def surf_row(maps, title, fname, threshold=3.1, vmax=None, cmap="cold_hot", views=("ventral", "lateral")):
    fig = plt.figure(figsize=(4.6 * len(views) * 2 + 1.5, 3.4 * len(maps)))
    k = 1
    for name, img in maps.items():
        for h in "LR":
            tex = to_surf(img, h)
            for view in views:
                ax = fig.add_subplot(len(maps), len(views) * 2, k, projection="3d"); k += 1
                plotting.plot_surf_stat_map(infl[h], tex, hemi=hemi[h], view=view, bg_map=sulc[h], threshold=threshold,
                                            vmax=vmax, cmap=cmap, colorbar=(view == views[-1] and h == "R"), axes=ax, figure=fig,
                                            title=f"{name}\n{hemi[h]} {view}", title_font_size=9)
                try:
                    plotting.plot_surf_contours(infl[h], roi_surf[h], levels=[1], colors=["k"], axes=ax, figure=fig)
                except Exception:
                    pass
    fig.suptitle(title, fontsize=13, weight="bold")
    fig.savefig(FIG / fname, dpi=130, bbox_inches="tight"); plt.show()
surf_row({f"{c} > others": tmaps[c] for c in cats}, "A. Category-vs-others t maps (localizer, whole brain; |t| > 3.1; outline = VVS ROI)",
         "4_category_contrasts_inflated.png", vmax=12)

## 5. Two items: weights, template, and what happens in working memory

Picks the face and the place item with the highest leave-one-item instance-out template match (validation). For each: its **weights** (item-vs-others t map from GLM B, inside its category mask from GLM A), its **template** (z-scored, lag-shifted localizer BOLD × weights), both in the VVS ROI on the inflated surface; then its **WM timecourses** -- the similarity of every WM volume after the image's view onset to its own template (target), to the other templates of its category (related) and to the other categories (others), split by the operation performed.

In [ ]:
category_masks, category_t, item_t, qc = T.feature_selection(cfg, sub_df, runs, roi_masker, items)
sets, occurrences = T.build_template_sets(cfg, tmpl_events, runs, items, category_masks, category_t, item_t)
ts = sets["item"]
val = pd.read_csv(HERE / "out/kim2020_denoised/sub-090/templates/sub-090_roi-vvs_template-validation.csv")
val = val[val["method"] == "kim2020"]
best = (val[val["comparison"] == "target"].groupby("item")["z"].mean().rename("z").reset_index()
        .merge(items, on="item"))
picks = [best[best["category"] == c].sort_values("z").iloc[-1]["item"] for c in ("face", "place")]
print("picked:", picks, "| validation z:", best.set_index("item").loc[picks, "z"].round(3).to_dict())
def roi_vec(values, mask):
    v = np.full(mask.shape, np.nan); v[mask] = values; return np.nan_to_num(v)
maps = {}
for ex in picks:
    m = ts.masks[ex]
    maps[f"{ex}: weights (t)"] = roi_masker.inverse_transform(roi_vec(ts.weights[ex][m], m))
    tpl = ts.templates[ex]
    maps[f"{ex}: template"] = roi_masker.inverse_transform(roi_vec(tpl / np.abs(tpl).max() * 10, m))
for k, img in maps.items():
    img.to_filename(FIG / ("sub-090_" + re.sub(r"[^A-Za-z0-9]+", "_", k) + ".nii.gz"))
surf_row(maps, "Item weights (GLM B, inside the category mask from GLM A) and templates (BOLD x weights, scaled to +-10)",
         "5_item_weights_templates.png", threshold=0.5, vmax=10, views=("ventral",))

In [ ]:
raw = pd.read_csv(HERE / "out/kim2020_denoised/group/fidelity/raw/roi-vvs_target-wmtimecourse_scores.csv", low_memory=False)
fig, axes = plt.subplots(1, len(picks) + 1, figsize=(17, 4.2), sharey=True)
for ax, ex in zip(axes, picks):
    d = raw[(raw["stimulus"] == ex) & raw["comparison"].isin(["target", "related", "others"])]
    for comp, c in zip(["target", "related", "others"], ["tab:red", "tab:orange", "tab:gray"]):
        s = d[d["comparison"] == comp].groupby("time_s")["z"].mean()
        ax.plot(s.index, s.values, color=c, label=comp, lw=2)
    ax.set_title(f"{ex} ({d['operation'].nunique()} operations, {d.drop_duplicates(['run_id', 'event_index']).shape[0]} trials)")
    ax.axvspan(0, 2.77, color="tab:orange", alpha=0.08); ax.axvspan(2.77, 5.53, color="tab:blue", alpha=0.08)
    ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("time from view onset (s)")
axes[0].set_ylabel("Fisher z (template vs. WM volume)"); axes[0].legend()
ax = axes[-1]
d = raw[raw["comparison"] == "target"]
for op, c in zip(["maintain", "suppress", "clear", "switch"], ["tab:green", "tab:purple", "tab:red", "tab:blue"]):
    s = d[d["operation"] == op].groupby("time_s")["z"].mean()
    ax.plot(s.index, s.values, color=c, label=op, lw=2)
ax.axvspan(0, 2.77, color="tab:orange", alpha=0.08); ax.axvspan(2.77, 5.53, color="tab:blue", alpha=0.08)
ax.axhline(0, color="k", lw=0.5); ax.set_title("all images: target similarity by operation"); ax.legend(); ax.set_xlabel("time from view onset (s)")
fig.suptitle("D-E. WM timecourses (shaded: image on screen, then operation cue; BOLD lags ~4-6 s)", weight="bold")
fig.tight_layout(); fig.savefig(FIG / "5b_item_timecourses.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# the representational structure the templates carry: template x template correlations (90 x 90),
# sorted by category > subcategory > image, using each template's own weights / mask (as scored)
order_ex = items.sort_values(["category", "subcategory", "item"])["item"].tolist()
def cell(a, b):
    """Mean r of image b's single localizer item instances with template a. On the diagonal the
    template is rebuilt without the item instance being scored (leave-one-item instance-out),
    so the same image is never correlated with itself."""
    m, w = ts.masks[a], ts.weights[a]
    rs = []
    for k, (_, pat) in enumerate(occurrences[b]):
        if a == b:
            loo = np.mean([p for j, (_, p) in enumerate(occurrences[a]) if j != k], axis=0)[m] * w[m]
            rs.append(ts.score(pat, a, template=loo))
        else:
            rs.append(ts.score(pat, a))
    return np.nanmean(rs)
R = np.array([[cell(a, b) for b in order_ex] for a in order_ex])
fig, ax = plt.subplots(figsize=(8.5, 7.5))
im = ax.imshow(R, cmap="RdBu_r", vmin=-0.4, vmax=0.4)
lab = items.set_index("item").loc[order_ex, "subcategory"].tolist()
edges = [i for i in range(1, len(lab)) if lab[i] != lab[i - 1]]
for e in edges: ax.axhline(e - 0.5, color="k", lw=0.6); ax.axvline(e - 0.5, color="k", lw=0.6)
mids = [(a + b) / 2 for a, b in zip([0] + edges, edges + [len(lab)])]
ax.set_xticks(mids); ax.set_xticklabels([lab[int(m)] for m in mids], rotation=45, ha="right", fontsize=8)
ax.set_yticks(mids); ax.set_yticklabels([lab[int(m)] for m in mids], fontsize=8)
ax.set_xlabel("pattern: single localizer item instances of the image"); ax.set_ylabel("template (weighted, masked)")
plt.colorbar(im, label="Pearson r"); ax.set_title("Template vs. single item instances (diagonal: leave-one-item instance-out)")
fig.tight_layout(); fig.savefig(FIG / "5c_template_similarity.png", dpi=150, bbox_inches="tight"); plt.show()
print("mean r -- same image:", np.diag(R).mean().round(3), "| same subcategory:",
      np.nanmean([R[i, j] for i in range(len(lab)) for j in range(len(lab)) if i != j and lab[i] == lab[j]]).round(3),
      "| different category:", np.nanmean([R[i, j] for i in range(len(lab)) for j in range(len(lab)) if lab[i].split('_')[-1] != lab[j].split('_')[-1]]).round(3))